# OSLLM: fine-tune Qwen3 on the process scheduling dataset

Runs on **Google Colab with a free T4 GPU** (Runtime → Change runtime type → T4 GPU).
It uses QLoRA through Unsloth: the base model is loaded in 4-bit and only small LoRA adapters are trained.

**Before you start:** on your laptop run `python finetune/build_dataset.py`, then upload
`finetune/data/train.jsonl` and `finetune/data/val.jsonl` when the upload cell asks.

At the end you download `osllm-ft.gguf`. Put it in `OSLLM/finetune/` and run `bash run.sh`;
the app then uses the fine-tuned model automatically.

In [ ]:
%%capture
!pip install unsloth

## 1. Settings
The 8B model gives the best answers. If Colab runs out of time or memory, use the 4B model or lower `MAX_TRAIN_EXAMPLES`.

In [ ]:
BASE_MODEL = "unsloth/Qwen3-8B"      # or "unsloth/Qwen3-4B" for faster training
MAX_SEQ_LEN = 4096                   # the longest examples in the dataset are about 3,600 tokens
MAX_TRAIN_EXAMPLES = None            # e.g. 3000 to train on a subset
EPOCHS = 1
LEARNING_RATE = 2e-4
LORA_RANK = 16

## 2. Upload the dataset

In [ ]:
import os
from google.colab import files
if not (os.path.exists("train.jsonl") and os.path.exists("val.jsonl")):
    uploaded = files.upload()        # choose train.jsonl and val.jsonl
print([f for f in os.listdir() if f.endswith(".jsonl")])

## 3. Load the base model in 4-bit and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

## 4. Format the conversations with the Qwen3 chat template

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
if MAX_TRAIN_EXAMPLES:
    data["train"] = data["train"].shuffle(seed=1).select(range(MAX_TRAIN_EXAMPLES))

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                     for m in batch["messages"]]}

data = data.map(to_text, batched=True, remove_columns=data["train"].column_names)
print(data)
print(data["train"][0]["text"][:1500])

## 5. Train
Only the assistant answers are trained on (the system prompt and the question are masked).

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=data["train"], eval_dataset=data["validation"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, warmup_steps=20,
        lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
        logging_steps=10, eval_strategy="steps", eval_steps=200,
        seed=3407, output_dir="outputs", report_to="none"))

trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

stats = trainer.train()
print(stats)

## 6. Quick check on a new problem

In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "You are OSLLM, the tutor for the Operating Systems course (BCSE303L)."},
    {"role": "user", "content": "Solve using SRTF. Processes: P1 (AT 0, BT 7), P2 (AT 2, BT 4), P3 (AT 4, BT 1), "
                                 "P4 (AT 5, BT 4). Find the average waiting time and turnaround time."}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                       enable_thinking=False).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=1200, temperature=0.1, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
# correct answer: average TAT 7, average WT 3

## 7. Save the LoRA adapter and export a GGUF file for Ollama

In [ ]:
import glob, shutil
model.save_pretrained("osllm-lora")
tokenizer.save_pretrained("osllm-lora")

model.save_pretrained_gguf("osllm-gguf", tokenizer, quantization_method="q4_k_m")
gguf = sorted(glob.glob("osllm-gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf"))
print(gguf)
shutil.copy(gguf[0], "osllm-ft.gguf")

In [ ]:
from google.colab import files
files.download("osllm-ft.gguf")   # about 5 GB for 8B, 2.5 GB for 4B

## 8. Use it on your laptop
1. Move `osllm-ft.gguf` into `OSLLM/finetune/`.
2. Run `bash run.sh` — it creates the `osllm-ft` model in Ollama and the app uses it.
3. Compare it with the base model on held-out problems:
```
python finetune/eval_model.py --model osllm --n 60
python finetune/eval_model.py --model osllm-ft --n 60
```